In [2]:
!pip install neo4j


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import sys
!{sys.executable} -m pip install neo4j


   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo4j]
   -------------------- ------------------- 1/2 [neo


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
from neo4j import GraphDatabase

# 1. Carrega o seu arquivo CSV do IMDB
csv_path = r"C:\Users\User\Documents\FATESG\Banco de dados não relacional\Código RAG\Data\imdb_top_1000.csv"
df = pd.read_csv(csv_path)

# 2. Configura a conexão com o Neo4j (as mesmas credenciais que você usou na tela)
NEO4J_URI = "bolt://localhost:7687"  # Porta de comunicação interna do Neo4j
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password123"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def criar_rede_cinema(tx, row):
    # Query em Cypher (a linguagem de consulta do Neo4j)
    # Ela cria o Filme, o Diretor, o Gênero e os conecta se não existirem (MERGE)
    query = """
    MERGE (f:Filme {titulo: $title})
    SET f.ano = $year, f.nota_imdb = $rating, f.resumo = $overview
    
    MERGE (d:Diretor {nome: $director})
    MERGE (g:Genero {nome: $genre})
    
    MERGE (d)-[:DIRIGIU]->(f)
    MERGE (f)-[:PERTENCE_AO_GENERO]->(g)
    """
    
    # Tratamento simples para pegar o primeiro gênero da lista (ex: "Drama, Romance" vira "Drama")
    primeiro_genero = str(row['Genre']).split(',')[0].strip()
    
    tx.run(query, 
           title=row['Series_Title'], 
           year=int(row['Released_Year']) if str(row['Released_Year']).isdigit() else 0,
           rating=float(row['IMDB_Rating']),
           overview=row['Overview'],
           director=row['Director'],
           genre=primeiro_genero)

# 3. Executa a carga linha por linha do CSV
print("Iniciando a carga dos filmes no Neo4j... Aguarde.")
with driver.session() as session:
    for index, row in df.iterrows():
        session.execute_write(criar_rede_cinema, row)

driver.close()
print("Carga concluída com sucesso! Seu mapa mental de filmes foi criado no Neo4j.")

Iniciando a carga dos filmes no Neo4j... Aguarde.
Carga concluída com sucesso! Seu mapa mental de filmes foi criado no Neo4j.
